# Qwen2-Audio baseline and int8 quantization


In [1]:
# # ============================================================
# # CELL 1: Imports and global config
# # ============================================================

# from pathlib import Path
# import shutil
# import csv
# import time

# import pandas as pd
# import soundfile as sf
# import torch
# from tqdm.auto import tqdm

# from transformers import (
#     AutoProcessor,
#     Qwen2AudioForConditionalGeneration,
#     BitsAndBytesConfig,
# )

# from comet import download_model, load_from_checkpoint


# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "acl60_60" / "2" / "acl_6060"
# FILTERED_ROOT = PROJECT_ROOT / "data" / "raw" / "acl6060_en_zh"
# MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
# OUT_DIR = PROJECT_ROOT / "outputs" / "qwen2audio_full_en_zh"

# DEV_MANIFEST = MANIFEST_DIR / "acl6060_dev_en_zh.csv"
# EVAL_MANIFEST = MANIFEST_DIR / "acl6060_eval_en_zh.csv"

# MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
# PROMPT_TEXT = "Translate this English speech into Chinese."

# DEVICE = "cuda:0"
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# print("project root exists:", PROJECT_ROOT.exists())
# print("raw root exists:", RAW_ROOT.exists())
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("gpu:", torch.cuda.get_device_name(0))


# # ============================================================
# # CELL 2: Build manifests if missing, then load full dataset
# # ============================================================

# def rebuild_en_zh_subset_and_manifests():
#     print("Rebuilding en->zh subset and manifests...")

#     if FILTERED_ROOT.exists():
#         shutil.rmtree(FILTERED_ROOT)

#     for split in ["dev", "eval"]:
#         (FILTERED_ROOT / split / "text" / "txt").mkdir(parents=True, exist_ok=True)
#         (FILTERED_ROOT / split / "segmented_wavs").mkdir(parents=True, exist_ok=True)

#         shutil.copy2(
#             RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
#             FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
#         )
#         shutil.copy2(
#             RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
#             FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
#         )
#         shutil.copytree(
#             RAW_ROOT / split / "segmented_wavs" / "gold",
#             FILTERED_ROOT / split / "segmented_wavs" / "gold",
#         )

#     MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

#     for split in ["dev", "eval"]:
#         en_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt"
#         zh_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt"
#         wav_dir = FILTERED_ROOT / split / "segmented_wavs" / "gold"

#         en_lines = en_file.read_text(encoding="utf-8").splitlines()
#         zh_lines = zh_file.read_text(encoding="utf-8").splitlines()
#         wavs = sorted(wav_dir.glob("sent_*.wav"), key=lambda p: int(p.stem.split("_")[1]))

#         assert len(en_lines) == len(zh_lines) == len(wavs), (
#             split, len(en_lines), len(zh_lines), len(wavs)
#         )

#         out_path = MANIFEST_DIR / f"acl6060_{split}_en_zh.csv"
#         with out_path.open("w", newline="", encoding="utf-8") as f:
#             writer = csv.writer(f)
#             writer.writerow(["id", "split", "audio_path", "source_en", "target_zh"])
#             for i, (wav, en, zh) in enumerate(zip(wavs, en_lines, zh_lines), start=1):
#                 writer.writerow([f"{split}_{i}", split, str(wav.relative_to(PROJECT_ROOT)), en, zh])

#         print("wrote", out_path, "rows:", len(wavs))


# if not DEV_MANIFEST.exists() or not EVAL_MANIFEST.exists():
#     rebuild_en_zh_subset_and_manifests()

# df = pd.concat(
#     [
#         pd.read_csv(DEV_MANIFEST),
#         pd.read_csv(EVAL_MANIFEST),
#     ],
#     ignore_index=True,
# )

# df["audio_path"] = df["audio_path"].apply(
#     lambda p: str((PROJECT_ROOT / p).resolve()) if not Path(p).is_absolute() else str(Path(p).resolve())
# )

# print("total rows:", len(df))
# print("first audio path:", df["audio_path"].iloc[0])
# print("first audio exists:", Path(df["audio_path"].iloc[0]).exists())

# df.head()


# # ============================================================
# # CELL 3: Tiny smoke subset
# # Change USE_FULL_DATASET to True when ready
# # ============================================================

# USE_FULL_DATASET = True
# SMOKE_N = 8

# run_df = df.copy() if USE_FULL_DATASET else df.head(SMOKE_N).copy()
# print("rows selected for run:", len(run_df))


# # ============================================================
# # CELL 4: Load processor, baseline model, and int8 model
# # ============================================================

# print("Loading processor...")
# processor = AutoProcessor.from_pretrained(MODEL_ID)

# print("Loading baseline fp16 model...")
# baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )
# baseline_model.eval()

# print("Loading int8 model...")
# bnb_config = BitsAndBytesConfig(load_in_8bit=True)
# quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# quant_model.eval()

# print("Models loaded.")


# # ============================================================
# # CELL 5: Inference helper with progress bar
# # ============================================================

# def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=128):
#     audio, sr = sf.read(str(audio_path))

#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)

#     conversation = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "audio", "audio_url": str(audio_path)},
#                 {"type": "text", "text": prompt_text},
#             ],
#         }
#     ]

#     text = processor.apply_chat_template(
#         conversation,
#         add_generation_prompt=True,
#         tokenize=False,
#     )

#     inputs = processor(
#         text=text,
#         audios=[audio],
#         sampling_rate=sr,
#         return_tensors="pt",
#     )

#     inputs = {
#         k: v.to(DEVICE) if hasattr(v, "to") else v
#         for k, v in inputs.items()
#     }

#     with torch.no_grad():
#         generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

#     generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

#     pred = processor.batch_decode(
#         generated_ids,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False,
#     )[0].strip()

#     return pred


# def run_inference(model, processor, df_input, run_name):
#     preds = []
#     start = time.time()

#     for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
#         audio_path = Path(row["audio_path"])
#         pred = generate_translation(model, processor, audio_path)
#         preds.append(pred)

#     elapsed = time.time() - start

#     out_df = df_input.copy()
#     out_df["prediction"] = preds
#     return out_df, elapsed


# # ============================================================
# # CELL 6: Run baseline and int8, save predictions
# # ============================================================

# print("Running baseline fp16...")
# baseline_df, baseline_time = run_inference(
#     baseline_model,
#     processor,
#     run_df,
#     run_name="baseline_fp16",
# )
# baseline_out = OUT_DIR / "baseline_full_preds.csv"
# baseline_df.to_csv(baseline_out, index=False, encoding="utf-8")
# print("baseline saved to:", baseline_out)
# print("baseline seconds:", round(baseline_time, 2))

# print("Running int8...")
# quant_df, quant_time = run_inference(
#     quant_model,
#     processor,
#     run_df,
#     run_name="quantized_int8",
# )
# quant_out = OUT_DIR / "quant8_full_preds.csv"
# quant_df.to_csv(quant_out, index=False, encoding="utf-8")
# print("int8 saved to:", quant_out)
# print("int8 seconds:", round(quant_time, 2))

# baseline_df[["id", "target_zh", "prediction"]].head(3)


# # ============================================================
# # CELL 7: Load COMET and score both runs
# # ============================================================

# print("Downloading/loading COMET...")
# comet_path = download_model("Unbabel/wmt22-comet-da")
# comet_model = load_from_checkpoint(comet_path)

# print("Preparing COMET data for baseline...")
# baseline_comet_data = [
#     {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
#     for _, r in tqdm(baseline_df.iterrows(), total=len(baseline_df), desc="comet_prep_baseline")
# ]

# print("Preparing COMET data for int8...")
# quant_comet_data = [
#     {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
#     for _, r in tqdm(quant_df.iterrows(), total=len(quant_df), desc="comet_prep_int8")
# ]

# print("Scoring baseline with COMET...")
# baseline_comet_out = comet_model.predict(baseline_comet_data, batch_size=4, gpus=1)
# baseline_comet = baseline_comet_out.system_score if hasattr(baseline_comet_out, "system_score") else baseline_comet_out[1]

# print("Scoring int8 with COMET...")
# quant_comet_out = comet_model.predict(quant_comet_data, batch_size=4, gpus=1)
# quant_comet = quant_comet_out.system_score if hasattr(quant_comet_out, "system_score") else quant_comet_out[1]

# print("baseline COMET:", baseline_comet)
# print("int8 COMET:", quant_comet)


# # ============================================================
# # CELL 8: Summary and save
# # ============================================================

# summary = pd.DataFrame(
#     [
#         {
#             "run": "baseline_fp16",
#             "rows": len(baseline_df),
#             "seconds": baseline_time,
#             "comet": baseline_comet,
#         },
#         {
#             "run": "quantized_int8",
#             "rows": len(quant_df),
#             "seconds": quant_time,
#             "comet": quant_comet,
#         },
#     ]
# )

# baseline_score = summary.loc[summary["run"] == "baseline_fp16", "comet"].iloc[0]
# summary["comet_diff_vs_baseline"] = summary["comet"] - baseline_score
# summary["speedup_vs_baseline"] = baseline_time / summary["seconds"]

# summary_out = OUT_DIR / "summary_comet_only.csv"
# summary.to_csv(summary_out, index=False)

# print("summary saved to:", summary_out)
# summary

In [2]:

from pathlib import Path
import json
import shutil
import time
import gc
import os

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
    BitsAndBytesConfig,
)
from comet import download_model, load_from_checkpoint


# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
PROMPT_TEXT = "Translate this English speech into Chinese."

# Keep defaults aligned with your current table rows
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "64"))
COMET_BATCH_SIZE = int(os.environ.get("COMET_BATCH_SIZE", "16"))
RUN_SPLIT = os.environ.get("RUN_SPLIT", "eval")  # eval or dev
FORCE_RESAVE_ARTIFACTS = os.environ.get("FORCE_RESAVE_ARTIFACTS", "0") == "1"

if RUN_SPLIT not in {"eval", "dev"}:
    raise ValueError(f"Unsupported RUN_SPLIT={RUN_SPLIT}")

MANIFEST = MANIFEST_DIR / f"acl6060_{RUN_SPLIT}_en_zh.csv"

EXPERIMENT_NAME = f"qwen2audio_{RUN_SPLIT}_only_en_zh_bnb_int8_maxnew{MAX_NEW_TOKENS}"
EXP_DIR = PROJECT_ROOT / "outputs" / "experiment_results" / EXPERIMENT_NAME
ARTIFACTS_DIR = EXP_DIR / "model_artifacts"
EXP_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0"

print("project root exists:", PROJECT_ROOT.exists())
print("manifest:", MANIFEST)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("experiment:", EXPERIMENT_NAME)


# ============================================================
# HELPERS
# ============================================================

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def dir_size_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except FileNotFoundError:
                pass
    return total


def ensure_clean_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def maybe_save_model_artifact(model, save_dir: Path, label: str):
    """
    Save a HuggingFace model artifact and return its size in bytes.
    If the artifact already exists and FORCE_RESAVE_ARTIFACTS=0, reuse it.
    """
    if save_dir.exists() and any(save_dir.iterdir()) and not FORCE_RESAVE_ARTIFACTS:
        size_bytes = dir_size_bytes(save_dir)
        print(f"{label}: reusing existing artifact at {save_dir} ({size_bytes / (1024**3):.3f} GB)")
        return size_bytes, "reused_saved_artifact"

    ensure_clean_dir(save_dir)
    print(f"{label}: saving model artifact to {save_dir} ...")
    # safe_serialization=False is the safest option here for BnB compatibility
    model.save_pretrained(save_dir, safe_serialization=False)
    size_bytes = dir_size_bytes(save_dir)
    print(f"{label}: artifact size = {size_bytes / (1024**3):.3f} GB")
    return size_bytes, "saved_artifact"


def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, processor, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        audio_path = Path(row["audio_path"])
        pred = generate_translation(model, processor, audio_path)
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]
    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    return comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]


# ============================================================
# DATA
# ============================================================

run_df = pd.read_csv(MANIFEST).copy()
run_df["audio_path"] = run_df["audio_path"].apply(
    lambda p: str((PROJECT_ROOT / p).resolve()) if not Path(p).is_absolute() else str(Path(p).resolve())
)

print("rows selected for run:", len(run_df))
print("first audio path:", run_df["audio_path"].iloc[0])
print("first audio exists:", Path(run_df["audio_path"].iloc[0]).exists())


# ============================================================
# LOAD PROCESSOR AND MODELS
# ============================================================

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading baseline fp16 model...")
baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()

print("Loading int8 model...")
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
quant_model.eval()

print("Models loaded.")


# ============================================================
# RECORD MODEL ARTIFACT SIZES BEFORE INFERENCE
# ============================================================

baseline_artifact_dir = ARTIFACTS_DIR / "baseline_fp16"
quant_artifact_dir = ARTIFACTS_DIR / "bitsandbytes_int8"

baseline_artifact_bytes, baseline_artifact_source = maybe_save_model_artifact(
    baseline_model, baseline_artifact_dir, "baseline_fp16"
)

quant_artifact_bytes = None
quant_artifact_source = None
quant_save_error = None

try:
    quant_artifact_bytes, quant_artifact_source = maybe_save_model_artifact(
        quant_model, quant_artifact_dir, "bitsandbytes_int8"
    )
except Exception as e:
    quant_save_error = repr(e)
    print("warning: failed to save BitsAndBytes artifact, leaving compressed size unavailable:")
    print(quant_save_error)

size_report = {
    "experiment_name": EXPERIMENT_NAME,
    "run_split": RUN_SPLIT,
    "max_new_tokens": MAX_NEW_TOKENS,
    "baseline_artifact_dir": str(baseline_artifact_dir),
    "baseline_artifact_bytes": int(baseline_artifact_bytes),
    "baseline_artifact_gb": baseline_artifact_bytes / (1024 ** 3),
    "baseline_artifact_source": baseline_artifact_source,
    "quant_artifact_dir": str(quant_artifact_dir),
    "quant_artifact_bytes": int(quant_artifact_bytes) if quant_artifact_bytes is not None else None,
    "quant_artifact_gb": (quant_artifact_bytes / (1024 ** 3)) if quant_artifact_bytes is not None else None,
    "quant_artifact_source": quant_artifact_source,
    "quant_save_error": quant_save_error,
}
size_report_path = EXP_DIR / "model_artifact_size_report.json"
with open(size_report_path, "w", encoding="utf-8") as f:
    json.dump(size_report, f, indent=2)
print("size report saved to:", size_report_path)


# ============================================================
# INFERENCE
# ============================================================

print("Running baseline fp16...")
baseline_df, baseline_time = run_inference(
    baseline_model,
    processor,
    run_df,
    run_name="baseline_fp16",
)
baseline_preds_out = EXP_DIR / "baseline_fp16_preds.csv"
baseline_df.to_csv(baseline_preds_out, index=False, encoding="utf-8")
print("baseline saved to:", baseline_preds_out)
print("baseline seconds:", round(baseline_time, 2))

print("Running int8...")
quant_df, quant_time = run_inference(
    quant_model,
    processor,
    run_df,
    run_name="quantized_int8",
)
quant_preds_out = EXP_DIR / "quantized_int8_preds.csv"
quant_df.to_csv(quant_preds_out, index=False, encoding="utf-8")
print("int8 saved to:", quant_preds_out)
print("int8 seconds:", round(quant_time, 2))


# ============================================================
# COMET
# ============================================================

print("Downloading/loading COMET...")
comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("Scoring baseline with COMET...")
baseline_comet = compute_comet(baseline_df, comet_model, batch_size=COMET_BATCH_SIZE)

print("Scoring int8 with COMET...")
quant_comet = compute_comet(quant_df, comet_model, batch_size=COMET_BATCH_SIZE)

print("baseline COMET:", baseline_comet)
print("int8 COMET:", quant_comet)


# ============================================================
# SUMMARY
# ============================================================

baseline_compression_ratio = 1.0
quant_compression_ratio = (
    (baseline_artifact_bytes / quant_artifact_bytes)
    if (quant_artifact_bytes is not None and quant_artifact_bytes > 0)
    else None
)

timestamp_utc = pd.Timestamp.utcnow().isoformat()

baseline_summary = pd.DataFrame(
    [
        {
            "timestamp_utc": timestamp_utc,
            "experiment_name": EXPERIMENT_NAME,
            "run": "baseline_fp16_eval",
            "split": RUN_SPLIT,
            "rows": len(baseline_df),
            "seconds": baseline_time,
            "seconds_per_item": baseline_time / len(baseline_df),
            "comet": baseline_comet,
            "baseline_comet_for_split": baseline_comet,
            "comet_delta_vs_baseline": 0.0,
            "model_id": MODEL_ID,
            "prompt_text": PROMPT_TEXT,
            "max_new_tokens": MAX_NEW_TOKENS,
            "comet_batch_size": COMET_BATCH_SIZE,
            "predictions_csv": str(baseline_preds_out),
            "quantization_method": "none",
            "precision": "fp16",
            "compression_method": "none",
            "original_total_bytes": int(baseline_artifact_bytes),
            "compressed_total_bytes": int(baseline_artifact_bytes),
            "compression_ratio": baseline_compression_ratio,
            "original_total_gb": baseline_artifact_bytes / (1024 ** 3),
            "compressed_total_gb": baseline_artifact_bytes / (1024 ** 3),
            "size_scope": "saved_model_artifact",
            "size_measurement_source": baseline_artifact_source,
            "model_artifact_dir": str(baseline_artifact_dir),
            "size_report_json": str(size_report_path),
        }
    ]
)

quant_summary = pd.DataFrame(
    [
        {
            "timestamp_utc": timestamp_utc,
            "experiment_name": EXPERIMENT_NAME,
            "run": "quantized_int8_eval",
            "split": RUN_SPLIT,
            "rows": len(quant_df),
            "seconds": quant_time,
            "seconds_per_item": quant_time / len(quant_df),
            "comet": quant_comet,
            "baseline_comet_for_split": baseline_comet,
            "comet_delta_vs_baseline": quant_comet - baseline_comet,
            "model_id": MODEL_ID,
            "prompt_text": PROMPT_TEXT,
            "max_new_tokens": MAX_NEW_TOKENS,
            "comet_batch_size": COMET_BATCH_SIZE,
            "predictions_csv": str(quant_preds_out),
            "quantization_method": "bitsandbytes_int8",
            "precision": "int8",
            "compression_method": "bitsandbytes_int8",
            "original_total_bytes": int(baseline_artifact_bytes),
            "compressed_total_bytes": int(quant_artifact_bytes) if quant_artifact_bytes is not None else None,
            "compression_ratio": quant_compression_ratio,
            "original_total_gb": baseline_artifact_bytes / (1024 ** 3),
            "compressed_total_gb": (quant_artifact_bytes / (1024 ** 3)) if quant_artifact_bytes is not None else None,
            "size_scope": "saved_model_artifact",
            "size_measurement_source": quant_artifact_source,
            "model_artifact_dir": str(quant_artifact_dir),
            "size_report_json": str(size_report_path),
            "size_measurement_error": quant_save_error,
        }
    ]
)

baseline_summary_path = EXP_DIR / "baseline_fp16_eval_summary.csv"
quant_summary_path = EXP_DIR / "quantized_int8_eval_summary.csv"
combined_summary_path = EXP_DIR / "summary_experiment.csv"

baseline_summary.to_csv(baseline_summary_path, index=False)
quant_summary.to_csv(quant_summary_path, index=False)
pd.concat([baseline_summary, quant_summary], ignore_index=True).to_csv(combined_summary_path, index=False)

print("baseline summary saved to:", baseline_summary_path)
print("int8 summary saved to:", quant_summary_path)
print("combined summary saved to:", combined_summary_path)

print("\nBaseline summary:")
print(baseline_summary.to_string(index=False))

print("\nInt8 summary:")
print(quant_summary.to_string(index=False))


# ============================================================
# CLEANUP
# ============================================================

del baseline_model
del quant_model
cleanup_cuda()


project root exists: True
manifest: /home/paperspace/projects/iwslt2026-compression/data/manifests/acl6060_eval_en_zh.csv
cuda available: True
gpu: NVIDIA RTX A6000
experiment: qwen2audio_eval_only_en_zh_bnb_int8_maxnew64
rows selected for run: 416
first audio path: /home/paperspace/projects/iwslt2026-compression/data/raw/acl6060_en_zh/eval/segmented_wavs/gold/sent_1.wav
first audio exists: True
Loading processor...
Loading baseline fp16 model...


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading int8 model...


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


...

Scoring baseline with COMET...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|█| 26/26 [00:02<00:00, 11
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Scoring int8 with COMET...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|█| 26/26 [00:02<00:00, 12


baseline COMET: 0.7791628381237388
int8 COMET: 0.77519199395409
baseline summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/baseline_fp16_eval_summary.csv
int8 summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/quantized_int8_eval_summary.csv
combined summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/summary_experiment.csv

Baseline summary:
                   timestamp_utc                              experiment_name                run split  rows    seconds  seconds_per_item    comet  baseline_comet_for_split  comet_delta_vs_baseline                     model_id                                 prompt_text  max_new_tokens  comet_batch_size                                                                                          